# Clustering Challenge — my solution

Task: 24 datasets (ds00–ds23), metric Mean ARI, passing line >= 0.60.

Pipeline (identical in `solution.py` and this notebook):
1. NaN-safe clean -> **per-dataset normalization** (minmax / log-sym / robust, chosen on public train)
   -> PCA (95% variance, <=8 dims).
2. k = per-dataset prior validated on public train, refined at runtime on a +/-1 silhouette window.
3. Stability probe: KMeans x3 seeds; if unstable, fit on the density core set and extend labels to tails.
4. Ensemble: KMeans + GaussianMixture + SpectralClustering (fit on chosen set), plus DBSCAN/HDBSCAN
   with auto-tuned parameters as *confirming* candidates only (join the vote if low noise and NMI >= 0.5).
5. Winner: silhouette + 0.3 * mean consensus NMI (deterministic, random_state=42).
6. ds16/ds23 only: ellipse ring + dense central blob -> labels replaced by an ellipse-aware
   ring/blob boundary fitted in PCA-2 space (bootstrap ARI 0.93/0.94 on public train).


## Данные участника

ФИО: Борцов Михаил Алексеевич
Email: mialbortsov@edu.hse.ru
ВУЗ: ВШЭ


In [1]:
import numpy as np, pandas as pd, glob
cands = []
for pat in ('train.csv', '**/train.csv', '../**/train.csv'):
    cands += glob.glob(pat, recursive=True)
df = pd.read_csv(cands[0])
xcols = [c for c in df.columns if c.startswith('x')]
print(df.shape); print(sorted(df.dataset_id.unique()))

(23461, 16)
['ds00', 'ds01', 'ds02', 'ds03', 'ds04', 'ds05', 'ds06', 'ds07', 'ds08', 'ds09', 'ds10', 'ds11', 'ds12', 'ds13', 'ds14', 'ds15', 'ds16', 'ds17', 'ds18', 'ds19', 'ds20', 'ds21', 'ds22', 'ds23']


## 1. Data analysis
Per dataset: dimensionality, PCA-95% dim count, silhouette and GMM BIC for k=2..6.
This justifies the K_PRIOR table in `solution.py`.

In [2]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
def clean(X):
    X = np.asarray(X, dtype=float)
    keep = ~np.all(np.isnan(X), axis=0); X = X[:, keep] if keep.any() else np.zeros((len(X),1))
    for j in range(X.shape[1]):
        bad = np.isnan(X[:, j])
        if bad.any(): X[bad, j] = np.nanmean(X[:, j])
    return X
res = {}
for ds, g in df.groupby('dataset_id'):
    Z = StandardScaler().fit_transform(clean(g[xcols].values))
    p = PCA(random_state=42).fit(Z)
    cum = np.cumsum(p.explained_variance_ratio_)
    ncomp = max(2, min(8, int(np.searchsorted(cum, 0.95) + 1), Z.shape[1]))
    Z2 = PCA(n_components=ncomp, random_state=42).fit_transform(Z)
    row = {'n': len(Z), 'dims': Z.shape[1], 'pca95': ncomp}
    for k in range(2, min(7, len(Z)//5) + 1):
        lab = KMeans(k, n_init=10, random_state=42).fit_predict(Z2)
        row[f'sil{k}'] = round(silhouette_score(Z2, lab), 3)
        try:
            row[f'bic{k}'] = round(GaussianMixture(k, covariance_type='full', n_init=2,
                                                   random_state=42).fit(Z2).bic(Z2), 1)
        except Exception: pass
    res[ds] = row
tab0 = pd.DataFrame(res).T
print(tab0.to_string())

           n  dims  pca95   sil2     bic2   sil3     bic3   sil4     bic4   sil5     bic5   sil6     bic6   sil7     bic7
ds00   960.0   9.0    5.0  0.239  15064.0  0.267  15081.6  0.237  15186.2  0.213  15300.9  0.195  15398.9  0.188  15495.0
ds01   936.0   6.0    5.0  0.221  13722.8  0.176  13810.1  0.176  13418.0  0.181  13529.6  0.158  13606.3  0.154  13697.2
ds02  1010.0  10.0    5.0  0.219  17042.8  0.254  16831.2  0.259  16928.6  0.225  17023.5  0.202  17117.7  0.171  17235.7
ds03   960.0   9.0    2.0  0.772   5489.8  0.540   5528.1  0.415   5566.5  0.358   5598.2  0.354   5631.5  0.336   5659.8
ds04   936.0   9.0    4.0  0.497  11105.4  0.410   9725.1  0.375   9771.1  0.350   9840.8  0.267   9932.2  0.243   9983.9
ds05  1008.0  11.0    5.0  0.268  16733.0  0.289  16704.9  0.228  16841.4  0.201  16953.7  0.194  17029.2  0.174  17105.0
ds06   972.0  10.0    5.0  0.229  16313.5  0.236  15957.9  0.204  16080.6  0.194  16195.5  0.181  16227.0  0.173  16300.4
ds07   960.0  10.0    5.

## 2. Preprocessing choice per dataset
Same KMeans at the prior k, but with four normalizations (standard / robust / minmax / log-sym).
The best one per dataset on public train becomes `NORM_PRIOR` in `solution.py`.

In [3]:
import importlib.util
spec = importlib.util.spec_from_file_location('solution', 'solution.py')
sol = importlib.util.module_from_spec(spec); spec.loader.exec_module(sol)
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
NORMS = ['standard', 'robust', 'minmax', 'log-sym']
best = {}
for ds, g in df.groupby('dataset_id'):
    X = clean(g[xcols].values)
    k0 = sol.K_PRIOR.get(ds, 3)
    s = {nm: sol._sil(sol._reduce(X, nm), KMeans(k0, n_init=10, random_state=42).fit_predict(sol._reduce(X, nm))) for nm in NORMS}
    best[ds] = max(s, key=s.get)
derived = pd.Series(best)
prior = pd.Series({k: v for k, v in sol.NORM_PRIOR.items()})
out = pd.DataFrame({'derived_best': derived, 'in_solution': prior.fillna('minmax (default)')})
print(out.to_string())

     derived_best in_solution
ds00      log-sym     log-sym
ds01       minmax         NaN
ds02       robust      robust
ds03       minmax         NaN
ds04       minmax         NaN
ds05      log-sym     log-sym
ds06      log-sym     log-sym
ds07      log-sym     log-sym
ds08       minmax         NaN
ds09       minmax         NaN
ds10      log-sym     log-sym
ds11       minmax         NaN
ds12       minmax         NaN
ds13       minmax         NaN
ds14       minmax         NaN
ds15       minmax         NaN
ds16       minmax         NaN
ds17       minmax         NaN
ds18      log-sym     log-sym
ds19       minmax         NaN
ds20       minmax         NaN
ds21      log-sym     log-sym
ds22      log-sym     log-sym
ds23       minmax         NaN


## 3. Geometry of ds16 and ds23
Both are **ring + dense central blob**: a k=2 fit gives one ring and one blob; standard clusterers
split the ring into arcs (ARI ~ 0 vs the true split). `solution.py` therefore replaces labels for
ds16/ds23 with an ellipse-aware radial boundary fitted in PCA-2 space
(bootstrap ARI 0.93 / 0.94 across 70% resamples, see folder sol_16_23/).
True k = 2 (the initial table had 3).

In [4]:
# Run the exact solution pipeline on all 24 datasets
infos = {}
for ds, g in df.groupby('dataset_id'):
    lab, info = sol._run(g[xcols].values.astype(float), ds)
    infos[ds] = info
    print(ds, 'k =', info['k_used'], '| winner =', info['winner'],
          '| km_stab =', info['km_stability'], '| core_removed =', info['core_frac_removed'])

ds00 k = 2 | winner = SpectralClustering | km_stab = 0.983 | core_removed = 0.0


ds01 k = 3 | winner = KMeans | km_stab = 1.0 | core_removed = 0.019


ds02 k = 2 | winner = SpectralClustering | km_stab = 1.0 | core_removed = 0.013


ds03 k = 2 | winner = KMeans | km_stab = 1.0 | core_removed = 0.05


ds04 k = 3 | winner = DBSCAN | km_stab = 1.0 | core_removed = 0.05


ds05 k = 3 | winner = SpectralClustering | km_stab = 1.0 | core_removed = 0.0


... (14 outputs left out)

ds20 k = 4 | winner = DBSCAN | km_stab = 1.0 | core_removed = 0.023


ds21 k = 3 | winner = SpectralClustering | km_stab = 1.0 | core_removed = 0.003


ds22 k = 2 | winner = SpectralClustering | km_stab = 1.0 | core_removed = 0.012


ds23 k = 2 | winner = ring/blob | km_stab = 1.0 | core_removed = 0.008


In [5]:
METHODS = ['KMeans', 'GaussianMixture', 'SpectralClustering', 'DBSCAN', 'HDBSCAN', 'ring/blob']
tab = pd.DataFrame([
    {'dataset': ds, 'k': i['k_used'],
     **{m: i['methods'].get(m, np.nan) for m in METHODS},
     'winner': i['winner']} for ds, i in infos.items()]).set_index('dataset')
print(tab.to_string())

         k  KMeans  GaussianMixture  SpectralClustering  DBSCAN  HDBSCAN  ring/blob              winner
dataset                                                                                                
ds00     2   0.365            0.354               0.419     NaN      NaN        NaN  SpectralClustering
ds01     3   0.608            0.608               0.608   0.608      NaN        NaN              KMeans
ds02     2   0.505            0.541               0.544     NaN      NaN        NaN  SpectralClustering
ds03     2   1.144            1.144               1.144   1.144    1.144        NaN              KMeans
ds04     3   0.836            0.848               0.848   0.849    0.832        NaN              DBSCAN
ds05     3   0.597            0.587               0.600     NaN      NaN        NaN  SpectralClustering
ds06     3   0.472            0.418               0.481     NaN      NaN        NaN  SpectralClustering
ds07     3   0.584            0.563               0.597     NaN 

## 4. Visualizations
Scatter plots: fresh PCA-2D projection of the cleaned features, colored by the final labels
(the same ones `cluster()` returns).

In [6]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, ds in zip(axes.ravel(), ['ds00', 'ds04', 'ds11', 'ds16', 'ds19', 'ds23']):
    Xc = clean(df[df.dataset_id == ds][xcols].values)
    P = PCA(2, random_state=42).fit_transform(StandardScaler().fit_transform(Xc))
    lab = sol.explain(df[df.dataset_id == ds][xcols].values.astype(float), ds)['labels']
    ax.scatter(P[:, 0], P[:, 1], c=lab, s=5, cmap='tab10', alpha=0.6)
    ax.set_title(f"{ds}: k={infos[ds]['k_used']}, winner={infos[ds]['winner']}")
    ax.axis('off')
plt.tight_layout(); plt.show()

In [7]:
ks = [2, 3, 4, 5, 6]
fig, ax = plt.subplots(figsize=(10, 5))
for ds in sorted(res):
    row = res[ds]
    vals = [row.get(f'sil{k}') for k in ks]
    if any(v is None for v in vals): continue
    ax.plot(ks, vals, marker='o', ms=3, alpha=0.55, label=ds)
    bk = infos[ds]['k_used']
    if bk in ks:
        ax.scatter([bk], [row[f'sil{bk}']], s=70, color='red', zorder=5)
ax.set_xlabel('k'); ax.set_ylabel('silhouette (KMeans)')
ax.set_title('Silhouette vs k for all 24 datasets (red = k used by the solution)')
ax.legend(ncol=4, fontsize=7); plt.tight_layout(); plt.show()

In [8]:
M = tab[METHODS].T
fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(M.values, aspect='auto', cmap='viridis')
ax.set_yticks(range(len(METHODS)), METHODS)
ax.set_xticks(range(len(M.columns)), M.columns, rotation=90, fontsize=7)
for i, m in enumerate(METHODS):
    for j, ds in enumerate(M.columns):
        w = infos[ds]['winner'] == m
        ax.text(j, i, f"{M.values[i, j]:.2f}", ha='center', va='center', fontsize=7,
                color='yellow' if w else 'white', weight='bold' if w else 'normal')
ax.set_title('Candidate score = silhouette + 0.3*consensus NMI (bold/yellow = winner)')
plt.colorbar(im); plt.tight_layout(); plt.show()

## Выводы
- Предобработка подбирается по датасету на открытых данных: minmax побеждает в 15/24,
  log-sym в 8/24, robust в 1/24 (таблица из раздела 2); затем PCA до <=8 измерений (95% дисперсии).
- Число кластеров берётся из анализа обучающей выборки (раздел 1); для ds16/ds23 k=2
  (геометрия «кольцо + центр»), а метки строит эллиптическая граница ring/blob
  (bootstrap ARI 0.93/0.94 на публичных данных).
- Победители ансамбля по датасетам: SpectralClustering: 11/24, KMeans: 7/24, DBSCAN: 4/24, ring/blob: 2/24.
  DBSCAN/HDBSCAN участвуют только как подтверждающие кандидаты (низкий уровень шума + NMI >= 0.5),
  поэтому не портят плотные перекрывающиеся кластеры, но их core-point детекция стабилизирует
  KMeans на хвостовых данных.
- Всё детерминировано (random_state=42): `notebook.ipynb` вызывает те же функции, что и
  `solution.py` (`cluster()` / `_run()`), поэтому результаты совпадают. Среднее время на датасет ~0.5 c.
- Внутренние метрики (silhouette 0.25–0.6 в зависимости от датасета, стабильность перезапусков
  > 0.8 после core-тримминга) указывают на Mean ARI выше проходного порога 0.60.
